# Leipzig Open Data API - First Look

This notebook runs basic CKAN API checks against the Leipzig Open Data portal and inspects the first 10 datasets in detail.

API base: `https://opendata.leipzig.de/api/3/action/`

> Note: In some local environments the certificate chain can fail. For exploration this notebook uses `verify=False`.

In [9]:
import json
import ssl
import urllib.parse
import urllib.request
from pprint import pprint

BASE = "https://opendata.leipzig.de/api/3/action"
SSL_CTX = ssl._create_unverified_context()

def ckan_call(action: str, params: dict | None = None):
    """Call a CKAN action endpoint and return parsed JSON."""
    query = urllib.parse.urlencode(params or {})
    url = f"{BASE}/{action}" + (f"?{query}" if query else "")
    with urllib.request.urlopen(url, timeout=30, context=SSL_CTX) as response:
        data = json.load(response)
    assert data.get("success") is True, f"CKAN call failed: {data.get('error')}"
    return data

print("Setup complete.")

Setup complete.


In [10]:
# Basic API checks

status = ckan_call("status_show")
print("CKAN version:", status["result"].get("ckan_version"))
print("Site title:", status["result"].get("site_title"))

search_meta = ckan_call("package_search", {"rows": 0})
total_datasets = search_meta["result"]["count"]
print("Total datasets in catalog:", total_datasets)

groups = ckan_call("group_list", {"all_fields": True})
orgs = ckan_call("organization_list", {"all_fields": True})
print("Number of groups:", len(groups["result"]))
print("Number of organizations:", len(orgs["result"]))

CKAN version: 2.10.7
Site title: Open Data-Portal der Stadt Leipzig
Total datasets in catalog: 395
Number of groups: 13
Number of organizations: 25


In [11]:
# Fetch first 10 datasets from package_search

first_ten = ckan_call("package_search", {"rows": 10, "start": 0})["result"]["results"]

print(f"Loaded {len(first_ten)} datasets (first page, 10 entries).")
print("Dataset IDs:")
for i, ds in enumerate(first_ten, start=1):
    print(f"{i:02d}. {ds['name']}")

Loaded 10 datasets (first page, 10 entries).
Dataset IDs:
01. stadtbezirksbudget-stadt-leipzig
02. geo-koordinaten-der-haltestellen-schulfahrplan-fahrbibliothek
03. geo-koordinaten-der-haltestellen-fahrbibliothek
04. luftreinhalteplan-prognose-strasse-2025-stadt-leipzig
05. oberburgermeisterwahlen-kleinraumig
06. namensvorschlage-strabenbenennungen
07. ergebnisse-oberbuergermeisterwahl-ortsteile
08. dauerzahlstellen-radverkehr-gesamtanzahl-der-fahrrader-pro-tag-der-letzten-2-jahre-stadt-leipzi
09. dauerzahlstellen-radverkehr-gesamtanzahl-der-fahrrader-pro-stunde-der-letzten-31-tage-stadt-lei
10. dauerzahlstellen-radverkehr-stationen-stadt-leipzig


In [12]:
# Call package_show for each of the first 10 datasets and output key fields

detailed = []

for i, ds in enumerate(first_ten, start=1):
    ds_id = ds["name"]
    data = ckan_call("package_show", {"id": ds_id})["result"]
    resource_formats = [
        (r.get("format") or "UNKNOWN").upper()
        for r in data.get("resources", [])
    ]

    record = {
        "index": i,
        "id": data.get("name"),
        "title": data.get("title"),
        "organization": (data.get("organization") or {}).get("title"),
        "groups": [g.get("display_name") for g in data.get("groups", [])],
        "tags": [t.get("name") for t in data.get("tags", [])],
        "resource_count": len(data.get("resources", [])),
        "resource_formats": sorted(set(resource_formats)),
        "license": data.get("license_title") or data.get("license_id"),
        "notes_preview": (data.get("notes") or "")[:200],
    }
    detailed.append(record)

# Output all 10 datasets
for item in detailed:
    print("=" * 90)
    print(f"[{item['index']:02d}] {item['title']} ({item['id']})")
    print("Organization:", item["organization"])
    print("Groups:", ", ".join(item["groups"]) if item["groups"] else "-")
    print("Tags:", ", ".join(item["tags"][:10]) if item["tags"] else "-")
    print("Resources:", item["resource_count"])
    print("Formats:", ", ".join(item["resource_formats"]) if item["resource_formats"] else "-")
    print("License:", item["license"])
    print("Notes preview:", item["notes_preview"] if item["notes_preview"] else "-")

print("=" * 90)
print("Finished: detailed output created for first 10 datasets.")

[01] Stadtbezirksbudget, Stadt Leipzig (stadtbezirksbudget-stadt-leipzig)
Organization: Amt für Geoinformation und Bodenordnung
Groups: Regierung und öffentlicher Sektor, Regionen und Städte
Tags: Fördermittel, Planung, Stadtbezirksbeiräte, lokale Gremien, opendata
Resources: 4
Formats: UNKNOWN, WFS
License: 
Notes preview: Der Datensatz enthält die folgenden Informationen über aus dem Stadtbezirksbudget geförderte Projekte und Vorschläge seit 2023. Mit der Einführung von Stadtbezirksbudgets möchte die Stadt Leipzig Part
[02] Geo-Koordinaten der Haltestellen Schulfahrplan Fahrbibliothek (geo-koordinaten-der-haltestellen-schulfahrplan-fahrbibliothek)
Organization: Städtische Bibliotheken
Groups: -
Tags: Bibliothek, Fahrbibliothek, Geodaten, Haltestellen, Koordinaten, Schule
Resources: 1
Formats: CSV
License: None
Notes preview: geografische Lage der Haltestellen der Fahrbibliothek an Leipziger Schulen, Koordinaten nach Gauß-Krüger, in Dezimalgrad und UTM (gülzig für das 1. Halbjahr 2026

In [13]:
# Optional: inspect one dataset JSON fully (change index 0..9)

dataset_index = 0
pprint(detailed[dataset_index])

{'groups': ['Regierung und öffentlicher Sektor', 'Regionen und Städte'],
 'id': 'stadtbezirksbudget-stadt-leipzig',
 'index': 1,
 'license': '',
 'notes_preview': 'Der Datensatz enthält die folgenden Informationen über aus '
                  'dem Stadtbezirksbudget geförderte Projekte und Vorschläge '
                  'seit 2023. Mit der Einführung von Stadtbezirksbudgets '
                  'möchte die Stadt Leipzig Part',
 'organization': 'Amt für Geoinformation und Bodenordnung',
 'resource_count': 4,
 'resource_formats': ['UNKNOWN', 'WFS'],
 'tags': ['Fördermittel',
          'Planung',
          'Stadtbezirksbeiräte',
          'lokale Gremien',
          'opendata'],
 'title': 'Stadtbezirksbudget, Stadt Leipzig'}


## Full Catalog Export: Metadata + First 5 Records per Dataset

This section loads all datasets from the Leipzig CKAN catalog, fetches full metadata (`package_show`) for each dataset, and tries to collect the first 5 rows via `datastore_search`.

- If a dataset has no datastore-enabled resource, row sample is marked as unavailable.
- Results are written to JSON files in the project root for further analysis.

In [14]:
# 1) Collect all dataset metadata

all_dataset_ids = ckan_call("package_list")["result"]
print(f"Total dataset IDs received: {len(all_dataset_ids)}")

all_metadata = []

for i, ds_id in enumerate(all_dataset_ids, start=1):
    try:
        ds = ckan_call("package_show", {"id": ds_id})["result"]
        all_metadata.append(ds)
    except Exception as exc:
        all_metadata.append({
            "name": ds_id,
            "_error": f"package_show failed: {exc}",
        })

    if i % 50 == 0 or i == len(all_dataset_ids):
        print(f"Metadata fetched: {i}/{len(all_dataset_ids)}")

print(f"Finished metadata fetch: {len(all_metadata)} datasets")

Total dataset IDs received: 413
Metadata fetched: 50/413
Metadata fetched: 100/413
Metadata fetched: 150/413
Metadata fetched: 200/413
Metadata fetched: 250/413
Metadata fetched: 300/413
Metadata fetched: 350/413
Metadata fetched: 400/413
Metadata fetched: 413/413
Finished metadata fetch: 413 datasets


In [15]:
# 2) For each dataset: get first 5 entries (best effort via datastore)

dataset_samples = []

for i, ds in enumerate(all_metadata, start=1):
    ds_id = ds.get("name", f"unknown_{i}")
    resources = ds.get("resources", []) if isinstance(ds, dict) else []

    # Find first datastore-enabled resource
    datastore_resource = None
    for r in resources:
        if r.get("datastore_active"):
            datastore_resource = r
            break

    sample_info = {
        "dataset_id": ds_id,
        "dataset_title": ds.get("title") if isinstance(ds, dict) else None,
        "resource_count": len(resources),
        "datastore_resource_id": datastore_resource.get("id") if datastore_resource else None,
        "sample_records": [],
        "sample_total": None,
        "sample_available": False,
        "note": None,
    }

    if datastore_resource is None:
        sample_info["note"] = "No datastore-enabled resource found"
    else:
        try:
            sample = ckan_call(
                "datastore_search",
                {"resource_id": datastore_resource["id"], "limit": 5},
            )["result"]
            sample_info["sample_records"] = sample.get("records", [])
            sample_info["sample_total"] = sample.get("total")
            sample_info["sample_available"] = True
        except Exception as exc:
            sample_info["note"] = f"datastore_search failed: {exc}"

    dataset_samples.append(sample_info)

    if i % 50 == 0 or i == len(all_metadata):
        print(f"Samples processed: {i}/{len(all_metadata)}")

print(f"Finished sample extraction: {len(dataset_samples)} datasets")

Samples processed: 50/413
Samples processed: 100/413
Samples processed: 150/413
Samples processed: 200/413
Samples processed: 250/413
Samples processed: 300/413
Samples processed: 350/413
Samples processed: 400/413
Samples processed: 413/413
Finished sample extraction: 413 datasets


In [16]:
# 3) Save outputs and print quick summary

with open("leipzig_all_datasets_metadata.json", "w", encoding="utf-8") as f:
    json.dump(all_metadata, f, ensure_ascii=False, indent=2)

with open("leipzig_all_datasets_first5_records.json", "w", encoding="utf-8") as f:
    json.dump(dataset_samples, f, ensure_ascii=False, indent=2)

available = sum(1 for x in dataset_samples if x.get("sample_available"))
missing = len(dataset_samples) - available

print("Saved:")
print("- leipzig_all_datasets_metadata.json")
print("- leipzig_all_datasets_first5_records.json")
print()
print("Summary:")
print(f"- Datasets total: {len(dataset_samples)}")
print(f"- With first-5 records available (datastore): {available}")
print(f"- Without datastore sample: {missing}")

Saved:
- leipzig_all_datasets_metadata.json
- leipzig_all_datasets_first5_records.json

Summary:
- Datasets total: 413
- With first-5 records available (datastore): 108
- Without datastore sample: 305


## Visual Exploration of Exported JSON Files

This section visualizes the exported files:

- `leipzig_all_datasets_metadata.json`
- `leipzig_all_datasets_first5_records.json`

No extra plotting libraries are required; charts are rendered as HTML bars directly in the notebook.

In [17]:
from collections import Counter, defaultdict
from IPython.display import HTML, display

with open("leipzig_all_datasets_metadata.json", "r", encoding="utf-8") as f:
    metadata = json.load(f)

with open("leipzig_all_datasets_first5_records.json", "r", encoding="utf-8") as f:
    samples = json.load(f)

print(f"Loaded metadata entries: {len(metadata)}")
print(f"Loaded sample entries:   {len(samples)}")

Loaded metadata entries: 413
Loaded sample entries:   413


In [18]:
# Build summary statistics

org_counter = Counter()
group_counter = Counter()
format_counter = Counter()
license_counter = Counter()
resources_per_dataset = []

for ds in metadata:
    if not isinstance(ds, dict) or ds.get("_error"):
        continue

    org = ((ds.get("organization") or {}).get("title") or "Unknown").strip() or "Unknown"
    org_counter[org] += 1

    groups = ds.get("groups", []) or []
    if groups:
        for g in groups:
            group_counter[(g.get("display_name") or g.get("name") or "Unknown Group").strip() or "Unknown Group"] += 1
    else:
        group_counter["(No group)"] += 1

    license_name = (ds.get("license_title") or ds.get("license_id") or "Unspecified").strip() or "Unspecified"
    license_counter[license_name] += 1

    resources = ds.get("resources", []) or []
    resources_per_dataset.append(len(resources))
    for r in resources:
        fmt = (r.get("format") or "UNKNOWN").strip().upper() or "UNKNOWN"
        format_counter[fmt] += 1

sample_available = sum(1 for s in samples if s.get("sample_available"))
sample_unavailable = len(samples) - sample_available

sample_size_counter = Counter(len(s.get("sample_records", [])) for s in samples)

sample_available_by_dataset = {s.get("dataset_id"): bool(s.get("sample_available")) for s in samples}
group_sample_stats = defaultdict(lambda: {"total": 0, "available": 0})
for ds in metadata:
    if not isinstance(ds, dict) or ds.get("_error"):
        continue
    ds_id = ds.get("name")
    groups = ds.get("groups", []) or [{"display_name": "(No group)"}]
    for g in groups:
        gname = (g.get("display_name") or g.get("name") or "(No group)").strip() or "(No group)"
        group_sample_stats[gname]["total"] += 1
        if sample_available_by_dataset.get(ds_id):
            group_sample_stats[gname]["available"] += 1

print("Summary statistics prepared.")

Summary statistics prepared.


In [19]:
# Helper visual functions (HTML-based charts)

def render_kpis(items):
    cards = "".join(
        f"""
        <div style='border:1px solid #ddd;border-radius:10px;padding:12px 14px;min-width:180px'>
            <div style='font-size:12px;color:#555'>{label}</div>
            <div style='font-size:24px;font-weight:700'>{value}</div>
        </div>
        """
        for label, value in items
    )
    display(HTML(f"<div style='display:flex;gap:10px;flex-wrap:wrap'>{cards}</div>"))


def render_bar_chart(counter, title, top_n=15, color="#2f6fed"):
    items = counter.most_common(top_n)
    if not items:
        display(HTML(f"<h4>{title}</h4><p>No data.</p>"))
        return

    max_val = max(v for _, v in items)
    rows = []
    for label, val in items:
        width = (val / max_val) * 100 if max_val else 0
        safe_label = str(label).replace("<", "&lt;").replace(">", "&gt;")
        rows.append(
            f"""
            <div style='display:grid;grid-template-columns: 320px 1fr 48px;gap:8px;align-items:center;margin:4px 0;'>
                <div style='font-size:12px;white-space:nowrap;overflow:hidden;text-overflow:ellipsis' title='{safe_label}'>{safe_label}</div>
                <div style='background:#eef3ff;border-radius:6px;height:20px;position:relative;'>
                    <div style='width:{width:.2f}%;background:{color};height:20px;border-radius:6px;'></div>
                </div>
                <div style='font-size:12px;text-align:right'>{val}</div>
            </div>
            """
        )

    html = f"<h4 style='margin:18px 0 8px'>{title}</h4>{''.join(rows)}"
    display(HTML(html))


def render_group_availability_table(group_stats, top_n=15):
    ranked = sorted(
        group_stats.items(),
        key=lambda kv: (kv[1]["available"] / kv[1]["total"] if kv[1]["total"] else 0),
        reverse=True,
    )[:top_n]

    rows = []
    for name, stats in ranked:
        total = stats["total"]
        available = stats["available"]
        rate = (available / total * 100) if total else 0
        rows.append(
            f"<tr><td>{name}</td><td>{available}</td><td>{total}</td><td>{rate:.1f}%</td></tr>"
        )

    table_html = f"""
    <h4 style='margin:18px 0 8px'>Top Groups by Sample Availability Rate</h4>
    <table style='border-collapse:collapse;width:100%;max-width:800px'>
      <thead>
        <tr>
          <th style='text-align:left;border-bottom:1px solid #ccc;padding:6px'>Group</th>
          <th style='text-align:right;border-bottom:1px solid #ccc;padding:6px'>With Sample</th>
          <th style='text-align:right;border-bottom:1px solid #ccc;padding:6px'>Total</th>
          <th style='text-align:right;border-bottom:1px solid #ccc;padding:6px'>Availability</th>
        </tr>
      </thead>
      <tbody>
        {''.join(rows)}
      </tbody>
    </table>
    """
    display(HTML(table_html))

print("Visualization helpers ready.")

Visualization helpers ready.


In [20]:
# Render dashboard-style visuals

kpi_items = [
    ("Datasets (metadata)", len(metadata)),
    ("Datasets (sample file)", len(samples)),
    ("Datasets with first-5 rows", sample_available),
    ("Datasets without sample", sample_unavailable),
    ("Unique organizations", len(org_counter)),
    ("Unique groups", len(group_counter)),
    ("Unique resource formats", len(format_counter)),
]
render_kpis(kpi_items)

render_bar_chart(org_counter, "Top Organizations by Dataset Count", top_n=15, color="#3366cc")
render_bar_chart(group_counter, "Top Groups by Dataset Count", top_n=15, color="#009688")
render_bar_chart(format_counter, "Top Resource Formats", top_n=20, color="#7b1fa2")
render_bar_chart(license_counter, "Top Licenses", top_n=10, color="#ef6c00")
render_bar_chart(sample_size_counter, "Distribution: Number of Rows Retrieved Per Dataset", top_n=6, color="#c2185b")

render_group_availability_table(group_sample_stats, top_n=15)

if resources_per_dataset:
    avg_resources = sum(resources_per_dataset) / len(resources_per_dataset)
    print(f"Average resources per dataset: {avg_resources:.2f}")
    print(f"Max resources in one dataset: {max(resources_per_dataset)}")
    print(f"Min resources in one dataset: {min(resources_per_dataset)}")

Group,With Sample,Total,Availability
"Justiz, Rechtssystem und öffentliche Sicherheit",1,1,100.0%
Internationale Themen,3,5,60.0%
(No group),71,176,40.3%
Regierung und öffentlicher Sektor,26,82,31.7%
Wirtschaft und Finanzen,4,39,10.3%
Bevölkerung und Gesellschaft,6,70,8.6%
Regionen und Städte,1,26,3.8%
Verkehr,1,27,3.7%
"Bildung, Kultur und Sport",1,37,2.7%
"Landwirtschaft, Fischerei, Forstwirtschaft und Nahrungsmittel",0,4,0.0%


Average resources per dataset: 1.93
Max resources in one dataset: 11
Min resources in one dataset: 0


## Record-Level View: First 5 Entries per Dataset

This section visualizes the sampled records for each dataset.

- Each dataset is shown as an expandable block.
- If sample rows are available, the first 5 records are rendered as a table.
- If not available, a note explains why (for example: no datastore-enabled resource).

In [21]:
# Helper to render record tables safely

def _escape_html(value):
    text = str(value)
    text = text.replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;")
    return text


def records_to_html_table(records, max_cols=12):
    if not records:
        return "<p style='margin:6px 0;color:#666'>No records returned.</p>"

    # Use union of keys (limited to max_cols for readability)
    all_keys = []
    seen = set()
    for rec in records:
        for k in rec.keys():
            if k not in seen:
                all_keys.append(k)
                seen.add(k)
    keys = all_keys[:max_cols]

    head = "".join(
        f"<th style='position:sticky;top:0;background:#f7f7f7;border:1px solid #ddd;padding:4px 6px;text-align:left'>{_escape_html(k)}</th>"
        for k in keys
    )

    body_rows = []
    for rec in records:
        cells = []
        for k in keys:
            v = rec.get(k, "")
            if isinstance(v, (dict, list)):
                v = json.dumps(v, ensure_ascii=False)
            cells.append(
                f"<td style='border:1px solid #eee;padding:4px 6px;vertical-align:top;max-width:260px;overflow:hidden;text-overflow:ellipsis;white-space:nowrap' title='{_escape_html(v)}'>{_escape_html(v)}</td>"
            )
        body_rows.append(f"<tr>{''.join(cells)}</tr>")

    return (
        "<div style='overflow:auto;max-width:100%;border:1px solid #eee;border-radius:8px'>"
        "<table style='border-collapse:collapse;font-size:12px;width:max-content;min-width:100%'>"
        f"<thead><tr>{head}</tr></thead>"
        f"<tbody>{''.join(body_rows)}</tbody>"
        "</table></div>"
    )

print("Record-table helper ready.")

Record-table helper ready.


In [22]:
# Render all datasets with expandable first-5 records

blocks = []
for item in samples:
    dataset_id = item.get("dataset_id", "unknown")
    title = item.get("dataset_title") or dataset_id
    sample_available = bool(item.get("sample_available"))
    resource_count = item.get("resource_count")
    sample_total = item.get("sample_total")
    note = item.get("note") or ""
    records = item.get("sample_records") or []

    status_color = "#1b8a3f" if sample_available else "#a33a3a"
    status_text = "sample available" if sample_available else "no sample"

    summary = (
        f"<div style='display:flex;gap:12px;flex-wrap:wrap;align-items:center'>"
        f"<span style='font-weight:600'>{_escape_html(title)}</span>"
        f"<span style='font-size:12px;color:#555'>({ _escape_html(dataset_id) })</span>"
        f"<span style='font-size:12px;color:{status_color};border:1px solid {status_color};border-radius:999px;padding:1px 8px'>{status_text}</span>"
        f"<span style='font-size:12px;color:#555'>resources: {resource_count}</span>"
        f"<span style='font-size:12px;color:#555'>sample_total: {sample_total if sample_total is not None else '-'} </span>"
        f"</div>"
    )

    if sample_available:
        content = records_to_html_table(records, max_cols=12)
    else:
        content = f"<p style='margin:8px 0;color:#666'>Reason: {_escape_html(note) if note else 'Not available'}</p>"

    block = (
        "<details style='margin:8px 0;padding:8px 10px;border:1px solid #ddd;border-radius:10px'>"
        f"<summary style='cursor:pointer'>{summary}</summary>"
        f"<div style='margin-top:8px'>{content}</div>"
        "</details>"
    )
    blocks.append(block)

html = "<h4>All datasets - first 5 entries (expand each row)</h4>" + "".join(blocks)
display(HTML(html))
print(f"Rendered {len(blocks)} dataset sample blocks.")

Rendered 413 dataset sample blocks.
